# Simulation des résultats du sondage sur l'efficacité du Pass Culture

Dans ce notebook, nous allons simuler les résultats du sondage mené auprès des utilisateurs du Pass Culture afin d'en évaluer l'efficacité.

Les bases de données utilisées ci-après ont été récupérées auprès de la Cour des Comptes : https://www.ccomptes.fr/fr/publications/premier-bilan-du-pass-culture. 

Tout d'abord, nous devons impoorter les éléments nécessaires.

In [13]:
from functions import *
import numpy as np
import pandas as pd


## Importation des données

Dans un premier temps, nous importons les 16 bases de données mises à disposition par la Cour des Comptes. Elles regroupent des statistiques agrégées et nous pouvons ainsi reproduire les figures présentées dans le rapport d'évaluation de la Cour.

In [14]:
dfs = {}  # dictionnaire pour stocker tous les DataFrames
for elt in ["C3", "C4", "G1", "G2", "G3", "G4", "G5", "G6", "G7", "G8", "G9", "G10", "G11", "G13", "G14", "G15"]:  # liste des df que l'on veut importer
    path = elt+".csv"
    dfs[f"df_{elt.lower()}"] = pd.read_csv(path, sep=";", encoding="latin-1") 
    # On ne peut pas utiliser importdata car l'encodage du fichier n'est pas le même

Nous avons donc obtenu 16 bases de données qui reprennent des statistiques descriptives sur le sondage à partir duquel le dispositif a été évalué. Nous devons maintenant simuler les données à partir de ces statistiques agrégées.

## Description des données à notre disposition

Dans un premier temps, nous allons récapituler les données dont nous disposons, puis nous passerons à la simulation. Pour créer le tableau suivant, nous avons utilisé la fonction "infosbase" sur chaque dataframe. 

<table>
  <caption>
    Données mises à disposition par la Cour des Comptes
  </caption>
  <thead>
    <tr>
      <th scope="col">Nom du data_frame</th>
      <th scope="col">Variables</th>
      <th scope="col">Nombre de lignes</th>
      <th scope="col">Description</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th scope="row">df_c3</th>
      <td>Département<br>Montant.moyen.dépensé.par.les.jeunes.du.département</td>
      <td>102 lignes</td>
      <td>Donne le montant moyen dépensé par les jeunes du département.</td>
    </tr>
    <tr>
      <th scope="row">df_c4</th>
      <td>Département<br>Score.de.diversification.moyen.par.département</td>
      <td>102 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g1</th>
      <td>Âge<br>15.+<br>16.+<br>17.+<br>18.+</td>
      <td>1 ligne</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g2</th>
      <td>Caractéristique<br>Valeur<br>Taux.d'activitation.du.pass</td>
      <td>12 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g3</th>
      <td>Statut.déclaré<br>Etudiant<br>Lycéen<br>Collégien<br>Apprenti,.alternant,.service.civique<br>Demandeur.d'emploi<br>Employé<br>Inactif</td>
      <td>1 ligne</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g4</th>
      <td>Trimestre<br>cat_agrr<br>prop_aggr</td>
      <td>66 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g5</th>
      <td>categories<br>Population<br>Pourcentage</td>
      <td>20 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g6</th>
      <td>trimestre<br>macro_rayon_r<br>prop_montant</td>
      <td>99 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g7</th>
      <td>categories<br>nombre_utilisateurs<br>Revenu</td>
      <td>42 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g8</th>
      <td>trimestre<br>Age.à.la.réservation<br>home<br>search</td>
      <td>20 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g9</th>
      <td>Origin<br>Composante.diversité<br>Prop</td>
      <td>10 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g10</th>
      <td>Origine<br>Catégorie<br>prop</td>
      <td>12 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g11</th>
      <td>Delta.de.diversification<br>Part.des.réservations</td>
      <td>6 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g13</th>
      <td>X1<br>LFI.2022<br>Exec.2022<br>LFI.2023<br>Exec.2023<br>LFI.2024<br>Exec.(prév.).2024</td>
      <td>2 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g14</th>
      <td>Service<br>Effectif</td>
      <td>6 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g15</th>
      <td>X1<br>16.ans<br>17.ans<br>18.ans<br>19.ans</td>
      <td>4 lignes</td>
      <td></td>
    </tr>
  </tbody>
</table>


In [15]:
infosbase(dfs["df_c3"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 102
Nombre de colonnes  : 1

NOMS DES COLONNES ET TYPES
Départements,"Montant.moyen.dépensé.par.les.jeunes.du.département"    object
dtype: object


In [16]:
print(dfs["df_c3"])

    Départements,"Montant.moyen.dépensé.par.les.jeunes.du.département"
0                                   01,"238.000000000"                
1                                   02,"238.000000000"                
2                                   03,"238.000000000"                
3                                   04,"230.000000000"                
4                                   05,"236.000000000"                
..                                                 ...                
97                                 972,"249.000000000"                
98                                 973,"238.000000000"                
99                                 974,"224.000000000"                
100                                975,"215.000000000"                
101                                976,"184.000000000"                

[102 rows x 1 columns]


In [17]:
infosbase(dfs["df_c4"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 102
Nombre de colonnes  : 1

NOMS DES COLONNES ET TYPES
Départements,"Score.de.diversification.moyen.par.département"    object
dtype: object


In [18]:
print(dfs["df_c4"])

    Départements,"Score.de.diversification.moyen.par.département"
0                                   01,11.916180534234           
1                                  02,11.6966770338844           
2                                   03,11.521567336187           
3                                  04,11.4996133023975           
4                                  05,13.2248186946011           
..                                                 ...           
97                                972,12.4236311239193           
98                                         973,13.7075           
99                                974,10.8825570980743           
100                                             975,15           
101                               976,12.7916666666667           

[102 rows x 1 columns]


In [19]:
infosbase(dfs["df_g1"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 1
Nombre de colonnes  : 1

NOMS DES COLONNES ET TYPES
Âge,"15.+","16.+","17.+","18.+"    object
dtype: object


In [20]:
print(dfs["df_g1"])

          Âge,"15.+","16.+","17.+","18.+"
0  Taux de couverture,0.48,0.62,0.74,0.82


In [21]:
infosbase(dfs["df_g2"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 12
Nombre de colonnes  : 1

NOMS DES COLONNES ET TYPES
Caractéristique,"Valeur","Taux.d'activitation.du.pass"    object
dtype: object


In [22]:
print(dfs["df_g2"])

   Caractéristique,"Valeur","Taux.d'activitation.du.pass"
0                      Sexe,"Femme",0.792976939241686    
1                      Sexe,"Homme",0.691525461124794    
2   Situation,"Travailleur ou Inactif",0.656367393...    
3              Situation,"Etudiant",0.812401214966635    
4       Origine sociale,"Populaire",0.679879992253948    
5         Origine sociale,"Moyenne",0.735280241669367    
6      Origine sociale,"Supérieure",0.807930656419016    
7   Taille de l'agglomération,"<2k",0.720481103109104    
8   Taille de l'agglomération,"2k-20k",0.736829555...    
9   Taille de l'agglomération,"20k-100k",0.7502696...    
10  Taille de l'agglomération,">100k",0.7516731410...    
11  Taille de l'agglomération,"Paris",0.7384459589...    


In [23]:
infosbase(dfs["df_g3"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 1
Nombre de colonnes  : 1

NOMS DES COLONNES ET TYPES
Statut.déclaré,"Etudiant","Lycéen","Collégien","Apprenti,.alternant,.service.civique","Demandeur.d'emploi","Employé","Inactif"    object
dtype: object


In [24]:
print(dfs["df_g3"])

  Statut.déclaré,"Etudiant","Lycéen","Collégien","Apprenti,.alternant,.service.civique","Demandeur.d'emploi","Employé","Inactif"
0  Nombre,1156653,2224717,249732,242931,108120,55...                                                                            


In [25]:
infosbase(dfs["df_g4"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 66
Nombre de colonnes  : 1

NOMS DES COLONNES ET TYPES
Trimestre,"cat_agrr","prop_aggr"    object
dtype: object


In [26]:
print(dfs["df_g4"])

               Trimestre,"cat_agrr","prop_aggr"
0                44378,"Autre",0.03462507748539
1   44378,"Spectacle vivant",0.0121202153342868
2          44378,"Art&Musée",0.0171044342713898
3             44378,"Musique",0.182411417605388
4         44378,"Audiovisuel",0.201917572620456
..                                          ...
61  45292,"Spectacle vivant",0.0153443054007485
62         45292,"Art&Musée",0.0187937801349768
63             45292,"Musique",0.27750201688678
64        45292,"Audiovisuel",0.208466517438264
65              45292,"Livre",0.427380008580463

[66 rows x 1 columns]


In [27]:
infosbase(dfs["df_g5"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 20
Nombre de colonnes  : 1

NOMS DES COLONNES ET TYPES
categories,"Population","Pourcentage"    object
dtype: object


In [28]:
print(dfs["df_g5"])

                categories,"Population","Pourcentage"
0                LIVRE,"Echantillon",72.7532075312799
1                 LIVRE,"15-17 ans ",39.7960310638639
2                CINEMA,"15-17 ans ",30.9665235486232
3               CINEMA,"Echantillon",50.4062853460063
4         MUSIQUE_LIVE,"Echantillon",20.1672396106866
5          AUDIOVISUEL,"Echantillon",19.7075587672132
6   MUSIQUE_ENREGISTREE,"Echantillon",18.006154810...
7   PRATIQUE ARTISTIQUE,"Echantillon",14.028219491...
8           AUDIOVISUEL,"15-17 ans ",7.09055125942278
9    MUSIQUE_ENREGISTREE,"15-17 ans ",5.7983729954842
10                 JEU,"Echantillon",8.08511932402353
11           SPECTACLE,"Echantillon",7.77164740022996
12               MUSEE,"Echantillon",6.17586476727634
13          MUSIQUE_LIVE,"15-17 ans ",3.4325007151329
14              AUTRES,"Echantillon",4.49539792804416
15                MUSEE,"15-17 ans ",2.26777731207439
16               AUTRES,"15-17 ans ",1.99455195637724
17  PRATIQUE ARTISTIQUE,"15-

In [29]:
infosbase(dfs["df_g6"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 99
Nombre de colonnes  : 1

NOMS DES COLONNES ET TYPES
trimestre,"macro_rayon_r","prop_montant"    object
dtype: object


In [30]:
print(dfs["df_g6"])

             trimestre,"macro_rayon_r","prop_montant"
0                     44378,"Arts",0.0250460379802202
1                    44378,"Autres",0.156261982105312
2         44378,"Bandes dessinées",0.0552619127442805
3         44378,"Sciences sociales",0.180702243978412
4                 44378,"Jeunesse",0.0744871556630176
..                                                ...
94                45292,"Jeunesse",0.0817605032896031
95              45292,"Littérature",0.309267645243188
96                    45292,"Manga",0.207224770824839
97        45292,"Poèsie & théâtre",0.0161230703675647
98  45292,"Religions, spiritualitées",0.0218838641...

[99 rows x 1 columns]


In [31]:
infosbase(dfs["df_g7"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 42
Nombre de colonnes  : 1

NOMS DES COLONNES ET TYPES
categories,"nombre_utilisateurs","Revenu"    object
dtype: object


In [32]:
print(dfs["df_g7"])

            categories,"nombre_utilisateurs","Revenu"
0    LIVRE,62773,"décile des revenus les plus élévés"
1   CINEMA,41362,"décile des revenus les plus élévés"
2   MUSIQUE_LIVE,16476,"décile des revenus les plu...
3     FILM,16195,"décile des revenus les plus élévés"
4   MUSIQUE_ENREGISTREE,13586,"décile des revenus ...
5   SPECTACLE,12598,"décile des revenus les plus é...
6     MUSEE,8038,"décile des revenus les plus élévés"
7       JEU,7571,"décile des revenus les plus élévés"
8   INSTRUMENT,5952,"décile des revenus les plus é...
9   BEAUX_ARTS,5183,"décile des revenus les plus é...
10    MEDIA,4220,"décile des revenus les plus élévés"
11  CONFERENCE,1200,"décile des revenus les plus é...
12  PRATIQUE_ART,1172,"décile des revenus les plus...
13  CARTE_JEUNES,19,"décile des revenus les plus é...
14             LIVRE,123572,"déciles intermédiaires "
15             CINEMA,82126,"déciles intermédiaires "
16       MUSIQUE_LIVE,36888,"déciles intermédiaires "
17               FILM,33952,

In [33]:
infosbase(dfs["df_g8"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 20
Nombre de colonnes  : 1

NOMS DES COLONNES ET TYPES
trimestre,"Age.à.la.réservation","home","search"    object
dtype: object


In [34]:
print(dfs["df_g8"])

     trimestre,"Age.à.la.réservation","home","search"
0   44378,"18-20 ans",0.0258034124905524,0.7510422...
1   44470,"18-20 ans",0.0312385401606633,0.7367099...
2   44562,"15-17 ans",0.0715229593060528,0.6767525...
3   44562,"18-20 ans",0.027474931840175,0.74015333...
4   44652,"15-17 ans",0.0572752080196249,0.6632662...
5   44652,"18-20 ans",0.0274775244954488,0.7309216...
6   44743,"15-17 ans",0.0563935293024851,0.6611011...
7   44743,"18-20 ans",0.0197226883881567,0.7269990...
8   44835,"15-17 ans",0.0788448881516929,0.6150054...
9   44835,"18-20 ans",0.0292061324722638,0.6574852...
10  44927,"15-17 ans",0.12524707291267,0.588800911...
11  44927,"18-20 ans",0.0471035045403614,0.6612424...
12  45017,"15-17 ans",0.122548173673891,0.58146212...
13  45017,"18-20 ans",0.0426084394395136,0.6418548...
14  45108,"15-17 ans",0.129734886277761,0.56701482...
15  45108,"18-20 ans",0.0430071558874984,0.6381444...
16  45200,"15-17 ans",0.155470656311438,0.51110166...
17  45200,"18-20 ans",0.0405

In [35]:
infosbase(dfs["df_g9"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 10
Nombre de colonnes  : 1

NOMS DES COLONNES ET TYPES
Origin,"Composante.diversité","Prop"    object
dtype: object


In [36]:
print(dfs["df_g9"])

                Origin,"Composante.diversité","Prop"
0       Page d'accueil,"Catégorie",0.306115929517232
1  Page d'accueil,"Sous-catégorie",0.406772417499486
2          Page d'accueil,"Format",0.207297550712374
3           Page d'accueil,"Lieux",0.367152302924966
4            Page d'accueil,"Genre",0.53097557001395
5           Recherche,"Catégorie",0.0999518749964243
6       Recherche,"Sous-catégorie",0.123399580765606
7              Recherche,"Format",0.0591019576418532
8                Recherche,"Lieux",0.221953990599827
9                Recherche,"Genre",0.322869241886102


In [37]:
infosbase(dfs["df_g10"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 12
Nombre de colonnes  : 1

NOMS DES COLONNES ET TYPES
Origine,"Catégorie","prop"    object
dtype: object


In [38]:
print(dfs["df_g10"])

                           Origine,"Catégorie","prop"
0            Page d'accueil,"Livre",0.177928218309768
1            Page d'accueil,"Autre",0.094550373588682
2      Page d'accueil,"Audiovisuel",0.565626157551894
3          Page d'accueil,"Musique",0.122685663649443
4   Page d'accueil,"Spectacle vivant",0.0152934988...
5       Page d'accueil,"Art&Musée",0.0239160880186259
6                 Recherche,"Livre",0.716512407457611
7                Recherche,"Autre",0.0401259477560861
8           Recherche,"Audiovisuel",0.139814834456625
9              Recherche,"Musique",0.0759144864635744
10   Recherche,"Spectacle vivant",0.00598916455394671
11           Recherche,"Art&Musée",0.0216431593121566


In [39]:
infosbase(dfs["df_g11"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 6
Nombre de colonnes  : 1

NOMS DES COLONNES ET TYPES
Delta.de.diversification,"Part.des.réservations"    object
dtype: object


In [40]:
print(dfs["df_g11"])

  Delta.de.diversification,"Part.des.réservations"
0                                         0,0.5246
1                                         1,0.2012
2                                         2,0.1128
3                                          3,0.025
4                                         4,0.0702
5                                         5,0.0663


In [41]:
infosbase(dfs["df_g13"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 2
Nombre de colonnes  : 1

NOMS DES COLONNES ET TYPES
X1,"LFI.2022","Exec.2022","LFI.2023","Exec.2023","LFI.2024","Exec.(prév.).2024"    object
dtype: object


In [42]:
print(dfs["df_g13"])

  X1,"LFI.2022","Exec.2022","LFI.2023","Exec.2023","LFI.2024","Exec.(prév.).2024"
0  Part individuelle,199,199.6,209.5,240.1,210.5,...                             
1                Part collective,45,18,51,51,62,80.2                             


In [43]:
infosbase(dfs["df_g14"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 6
Nombre de colonnes  : 1

NOMS DES COLONNES ET TYPES
Service,"Effectif"    object
dtype: object


In [44]:
print(dfs["df_g14"])

                                  Service,"Effectif"
0                             Direction technique,46
1                                              SG,19
2  Direction du pilotage (Direction des opération...
3                   Direction de la communication,17
4                               Direction produit,17
5                      Direction du développement,51


In [45]:
infosbase(dfs["df_g15"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 4
Nombre de colonnes  : 1

NOMS DES COLONNES ET TYPES
X1,"16.ans","17.ans","18.ans","19.ans"    object
dtype: object


In [46]:
print(dfs["df_g15"])

  X1,"16.ans","17.ans","18.ans","19.ans"
0                          2020,,,0,0.02
1                       2021,,,0.09,0.34
2               2022,0.19,0.25,0.39,0.49
3               2023,0.33,0.45,0.57,0.63


## Simulation 

Objectif :

Simuler une base de données au niveau individuel à partir de statistiques agrégées (moyennes, proportions, tableaux croisés)
publiées dans un rapport (ici : Cour des comptes, Pass Culture).
 
Structure du script :

1. Fondements théoriques : explication de la théorie mathématique sur laquelle se fonde la simulation, justification du choix de la méthode ;

2. Configuration générale (graine aléatoire, taille de l'échantillon)

3. Fonction d'ajustement proportionnel itératif (IPF / raking)
       -> pour caler une table jointe sur plusieurs marges connues

4. Fonction de validation : on ré-agrège les données simulées et on
       les compare aux statistiques d'origine

Les fonctions à créer seront à nouveaux définies dans le fichier functions.py afin de fluidifier la lecture du notebook.

### Fondements théoriques

#### Problème : inférence écologique

La désagrégation statistique répond à un problème d'inférence statistique, ou inférence écologique. On dispose de données agrégées (moyennes, pourcentages, totaux par groupes), et on souhaite reconstituer une base de données au niveau individuel, alors qu'on ne dispose pas des données à ce niveau.

L'objectif n'est ainsi pas de retrouver les "vraies" données, mais de simuler une base de données qui aurait mené aux mêmes résultats agrégés. Nous sommes donc bien dans une situation de simulation sous contrainte, et non de reconstruction.

L'inférence est ici dite "stochastique car la désagrégation repose sur des tirages aléatoires (type Monte Carlo <mark>à développer / préciser</mark>), plutôt que sur une règle déterministe. La valeur prise par une certaine variable est tirée aléatoirement pour chaque individu, afin de conserver l'aléa et la variabilité naturels que l'on observerait dans un véritable échantillon.

#### Difficultés principales 

La difficulté principale pour notre simulation est que les lois marginales ne permettent pas de déduire la loi jointe. 

Dans certains cas, l'hypothèse la plus simple est de supposer l'indépendance des variables. Chaque variable sera alors simulée séparément, indépendamment des autres. Cette solution est rapide est simple, mais elle peut introduire un biais lorsqu'en réalité, les variables sont corrélées.

Lorsque nous disposons de tableaux qui croisent déjà plusieurs variables, nous pourrons nous passer de l'hypothèse d'indépendance. En effet, nous avons alors des informations sur la loi jointe des différentes variables proposées, et nous pourrons donc les simuler ensemble au lieu de les simuler indépendamment. Cette option sera préférable lorsqu'elle est possible.

Ces deux options sont les deux pôles à partir desquels nous allons effectuer la simulation. 


### Configuration générale

La configuration est une étape importante car elle garantit la reproductibilité des résultats.

In [47]:
seed = 42             # graine aléatoire -> reproductibilité des simulations
nb_indiv = 10000      # taille de l'échantillon simulé (à ajuster)
 
rng = np.random.default_rng(seed)

### Fonction d'ajustement proportionnel

L'ajustement proportionnel itératif (*Iterative Proportional Fitting*, IPF), aussi appelé **raking**, 
permet de construire une table jointe compatible avec plusieurs lois marginales connues, sans jamais 
observer directement la loi jointe.

**Principe :**

1. On part d'une table initiale (par exemple sous hypothèse d'indépendance : le produit des marges normalisées) ;
2. À chaque itération, pour chaque variable à caler, on recalcule la marge courante de la table simulée et on la compare à la marge cible (issue des tableaux de la Cour des comptes) ;
3. On multiplie chaque ligne par le ratio (marge cible / marge courante) correspondant à sa modalité ;
4. On répète pour toutes les variables jusqu'à convergence (écart maximal entre marges courantes et marges cibles inférieur à un seuil `tol`).

Cet algorithme converge (sous des conditions assez générales) vers une table qui respecte toutes les marges fournies, tout en restant "la plus proche possible" de la table initiale au sens de la divergence de Kullback-Leibler. C'est la méthode standard de calage sur marges utilisée en statistique officielle (post-stratification, calage d'enquêtes).

### Validation

Une fois la base individuelle simulée, il est indispensable de vérifier qu'elle reproduit bien les 
statistiques agrégées de départ : c'est la seule façon de s'assurer que la simulation n'a pas introduit de 
biais. Pour cela, on ré-agrège les données simulées (calcul des proportions ou effectifs par modalité) et on 
compare ces valeurs aux marges cibles extraites des tableaux de la Cour des comptes.